In [51]:
import os
import requests
import csv
import pandas as pd
import time

API_KEY = os.getenv("GOOGLE_PLACES_API_KEY")

In [9]:
PLACES_SEARCH_URL = "https://places.googleapis.com/v1/places:searchText"
PLACE_DETAILS_URL = "https://places.googleapis.com/v1/places/{place_id}"

In [12]:
OUTPUT_FILE = "../datasets/university_reviews_slice_2.csv"

In [55]:
row_range = slice(1500, 1899)
unis = pd.read_csv("../datasets/us_names.csv", header=None, names=["name"])["name"].iloc[row_range]

In [56]:
print(unis)

1500    Herkimer County Community College
1501                      Hilbert College
1502        Hobart William Smith Colleges
1503                   Hofstra University
1504                  Houghton University
                      ...                
1894                   Cameron University
1895            Carl Albert State College
1896       University of Central Oklahoma
1897                Connors State College
1898              East Central University
Name: name, Length: 399, dtype: str


In [57]:
with open(OUTPUT_FILE, "a", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)

    for uni in unis:
        try:
            # Find place
            search = requests.post(
                PLACES_SEARCH_URL,
                headers={
                    "Content-Type": "application/json",
                    "X-Goog-Api-Key": API_KEY,
                    "X-Goog-FieldMask": "places.id,places.displayName,places.reviews"
                },
                json={"textQuery": uni}
            ).json()

            if not search.get("places"):
                writer.writerow([uni, []])
                continue

            place = search["places"][0]
            place_id = place["id"]
            matched_name = place["displayName"]["text"]
            reviews = place

            # Get reviews
            # details = requests.get(
            #     PLACE_DETAILS_URL.format(place_id=place_id),
            #     headers={
            #         "Content-Type": "application/json",
            #         "X-Goog-Api-Key": API_KEY,
            #         "X-Goog-FieldMask": "reviews"
            #     }
            # ).json()

            reviews = [r["text"]["text"] for r in place.get("reviews", []) if "text" in r]
            writer.writerow([uni, reviews])
            print(f"✓ {uni} ({len(reviews)} reviews)")

        except Exception as e:
            print(f"✗ {uni}: {e}")
            writer.writerow([uni, []])

        time.sleep(0.05)  # small delay to be safe


✓ Herkimer County Community College (5 reviews)
✓ Hilbert College (5 reviews)
✓ Hobart William Smith Colleges (5 reviews)
✓ Hofstra University (5 reviews)
✓ Houghton University (5 reviews)
✓ Hudson Valley Community College (5 reviews)
✓ Iona University (5 reviews)
✓ Island Drafting and Technical Institute (5 reviews)
✓ Ithaca College (5 reviews)
✓ Jamestown Community College (5 reviews)
✓ Jefferson Community College (5 reviews)
✓ Jewish Theological Seminary of America (2 reviews)
✓ The Juilliard School (5 reviews)
✓ Kehilath Yakov Rabbinical Seminary (0 reviews)
✓ Keuka College (5 reviews)
✓ LIM College (5 reviews)
✓ Le Moyne College (5 reviews)
✓ Long Island University (5 reviews)
✓ Machzikei Hadath Rabbinical College (0 reviews)
✓ Mandl School-The College of Allied Health (5 reviews)
✓ Manhattan University (5 reviews)
✓ Manhattan School of Music (5 reviews)
✓ Manhattanville University (5 reviews)
✓ Maria College of Albany (5 reviews)
✓ Marion S Whelan School of Nursing of Geneva Gene